# 🤖 LangChain Chat Models, Prompt Templates & LCEL Guide

Welcome to the **LangChain Gen AI & LCEL** guide! This notebook demonstrates the core building blocks of LangChain: configuring chat models, building flexible prompt templates, and composing chains using **LangChain Expression Language (LCEL)**.

---

### 🧠 Key Concepts & Objectives

LangChain provides modular abstractions to connect LLMs into production-grade pipelines:
- **Chat Models (ChatOpenAI)**: Interacting with state-of-the-art conversational LLMs.
- **Prompt Templates (ChatPromptTemplate)**: Structuring reusable prompts with system roles, user inputs, and dynamic variables.
- **LCEL (| Pipe Operator)**: Declaratively chaining prompts, models, and parsers together with automatic streaming, batching, and async support.
- **Output Parsers (StrOutputParser)**: Extracting clean string content directly from the LLM's raw message responses.
- **Observability (LangSmith)**: Tracing and monitoring execution steps via environment flags.

## 🔑 Step 1: Environment Configuration & LangSmith Tracing

Load the required API keys from the .env file:
- OPENAI_API_KEY: Authenticates calls to OpenAI models.
- LANGCHAIN_API_KEY & LANGCHAIN_TRACING_V2: Enables LangSmith tracing for debugging and logging pipeline invocations.

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPEN_AI_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

## 🚀 Step 2: Initialize the ChatOpenAI Model

Instantiate ChatOpenAI with your desired model (e.g. gpt-5.4-mini).

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4-mini')
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001A8FB40FC50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A8FB4146D0>, root_client=<openai.OpenAI object at 0x000001A8F8FE9D90>, root_async_client=<openai.AsyncOpenAI object at 0x000001A8FB401A10>, model_name='gpt-5.4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

## 💬 Step 3: Direct Model Invocation

Call the model directly using llm.invoke() with a natural language string prompt.

In [28]:
response = llm.invoke("tell me about quantum physics in 50 words, like you're explaining the same to someone")

## 📄 Step 4: Inspecting Model Output

The model returns an AIMessage object containing metadata and content. Use .content to view the response text.

In [29]:
print(response.content)

Quantum physics studies how tiny things like atoms and particles behave. Unlike everyday objects, they can act like both particles and waves, exist in uncertain states, and influence each other in surprising ways. It helps explain light, chemistry, electronics, and much of the modern world around us.


## 📝 Step 5: Creating Structured Prompts with ChatPromptTemplate

Define a multi-role prompt template using ChatPromptTemplate.from_messages:
- **System Message**: Establishes the AI's persona, expertise, and constraints.
- **User Message**: Accepts dynamic variables (e.g., {input}) passed during execution.

In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    [
        ("system", "You're an experienced and expert psychological researcher and professor, you have to explain the concepts asked by the user in a simple way."),
        ("user", "{input}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You're an experienced and expert psychological researcher and professor, you have to explain the concepts asked by the user in a simple way."), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

## 🔗 Step 6: Composing Chains with LCEL (prompt | llm)

Connect the prompt template to the LLM using the pipe operator (|). In invoking the chain, provide the dictionary of input variables.

In [ ]:
chain = prompt|llm

response = chain.invoke({
    "input":"Tell me what it OCD with symptoms and an example, in 100 words."
})

## 👁️ Step 7: Accessing Response from the Chain

Extract .content from the AIMessage returned by the chain.

In [34]:
response.content

'Obsessive-Compulsive Disorder (OCD) is a mental health condition where a person has unwanted, repeated thoughts, images, or urges called **obsessions**, and feels driven to do certain actions called **compulsions** to reduce anxiety. Common symptoms include repeated handwashing, checking things many times, needing things to be arranged in a certain way, or fear of harm, germs, or making mistakes. These thoughts and behaviors can take up a lot of time and interfere with daily life. **Example:** A person may fear contamination and wash their hands 20 times a day, even when they know their hands are already clean.'

## 🧹 Step 8: Adding StrOutputParser for Clean String Outputs

Add StrOutputParser() to the end of the LCEL chain (prompt | llm | parser). This automatically parses the output into a plain Python string.

In [37]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
chain = prompt|llm|parser

response = chain.invoke({
    "input":"Tell me what it stockholm syndrome with symptoms and an example, in 150 words."
})

## 📊 Step 9: Displaying the Final Parsed Output

Print the clean, parsed string output produced by the end-to-end chain.

In [40]:
print(response)

Stockholm syndrome is a psychological response in which a person who is kidnapped, abused, or held under control starts to feel sympathy, loyalty, or even affection toward the person hurting them. It can happen because the victim may believe the captor is the only one who can keep them safe, especially during fear, isolation, or dependency.

Common symptoms include:
- Positive feelings toward the abuser
- Defending the abuser’s actions
- Fear of police, rescue workers, or family
- Difficulty seeing the abuse as harmful
- Gratitude for small acts of kindness from the abuser

Example: A hostage is kept in a room for days. The captor gives them food and speaks kindly sometimes. Over time, the hostage may begin to trust the captor, feel sorry for them, or refuse help from police.

It is not an official mental disorder, but a possible trauma response.
